# Chapter 3 — Multicollinearity: Detection & Solutions
## University of Neuchâtel — Python for Econometrics

---

## What you will learn

| Step | What | How |
|---|---|---|
| 1 | **Understand** multicollinearity | Intuition + simulation |
| 2 | **Detect** it | Correlation, VIF, condition number |
| 3 | **Solve** it | Drop variable, combine, Ridge regression |
| 4 | **Verify** the fix | Re-run diagnostics, compare SEs |

---

**Format:** One cell per step. Manual calculation first, then statsmodels/numpy, then `np.isclose()` to verify.

## 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
import matplotlib.pyplot as plt
import wooldridge as woo
import os

In [ ]:
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)
print('Working directory:', chapter_dir)

---
## 1 — What is Multicollinearity?

### 1.1 Definition

**Multicollinearity** arises when two or more regressors are **highly linearly correlated**.

- **Perfect multicollinearity:** one regressor is an **exact** linear combination of others
  → $(X'X)$ is singular → OLS **cannot be computed** at all.
- **Imperfect (near) multicollinearity:** regressors are highly (but not perfectly) correlated
  → OLS can be computed, but SEs become **very large**.

### 1.2 Intuition

Imagine trying to separately estimate the effect of height in cm ($x_1$) and height in inches ($x_2$)
on weight. Since $x_2 = 2.54 \cdot x_1$, you can't separate their individual effects.
The data give you no information about which one 'caused' the change.

More realistically: if `educ` and `IQ` are highly correlated, it's hard for the data to tell us
whether high wages come from more education or higher IQ.

### 1.3 The Formula Shows Why SEs Blow Up

$$Var(\hat{\beta}_j) = \frac{\sigma^2}{SST_j \cdot (1 - R^2_j)}$$

where $R^2_j$ is the R² from regressing $x_j$ on all other regressors.

- As $R^2_j \to 1$ (perfect multicollinearity): $Var(\hat{\beta}_j) \to \infty$
- High $R^2_j$: SE of $\hat{\beta}_j$ **inflates** → wide confidence intervals → hard to reject $H_0$

### 1.4 What Multicollinearity Does NOT Do

| Claim | True? |
|---|---|
| OLS is biased | ✗ No — OLS remains unbiased |
| OLS is inconsistent | ✗ No — still consistent |
| Individual t-stats are small | ✓ Yes — SEs blow up |
| Overall F-stat is large | ✓ Often yes — joint significance can remain |
| Predictions are bad | ✗ No — predictions for in-sample x remain fine |

---
## 2 — Simulation: Seeing the Effect of Multicollinearity

We simulate data with varying degrees of correlation between $x_1$ and $x_2$,
and observe how the standard error of $\hat{\beta}_1$ changes.

### Step 2.1 — Simulate three datasets: low / moderate / high collinearity

In [ ]:
np.random.seed(42)
n = 200
true_b1 = 2.0
true_b2 = 3.0
results_sim = []
for rho, label in [(0.1, 'Low (r=0.1)'), (0.7, 'Moderate (r=0.7)'), (0.99, 'High (r=0.99)')]:
    # Correlated x1, x2
    Sigma = np.array([[1, rho], [rho, 1]])
    X_corr = np.random.multivariate_normal([0,0], Sigma, n)
    x1, x2 = X_corr[:,0], X_corr[:,1]
    u = np.random.normal(0, 1, n)
    y = 1 + true_b1*x1 + true_b2*x2 + u
    X = np.column_stack([np.ones(n), x1, x2])
    beta = np.linalg.solve(X.T@X, X.T@y)
    e    = y - X@beta
    sig2 = np.sum(e**2) / (n-3)
    V    = sig2 * np.linalg.inv(X.T@X)
    SE   = np.sqrt(np.diag(V))
    results_sim.append({'rho': rho, 'label': label,
                        'b1': beta[1], 'SE_b1': SE[1],
                        'b2': beta[2], 'SE_b2': SE[2]})
df_sim = pd.DataFrame(results_sim)
print(df_sim[['label','b1','SE_b1','b2','SE_b2']].to_string(index=False))

### Step 2.2 — Visualise: SE explodes as correlation rises

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,4))
rhos = df_sim['rho'].values
axes[0].bar(df_sim['label'], df_sim['SE_b1'], color=['green','orange','red'], edgecolor='black')
axes[0].axhline(df_sim['SE_b1'].iloc[0], color='gray', linestyle='--', label='SE at low correlation')
axes[0].set_ylabel('SE of b1_hat')
axes[0].set_title('Standard Error of b1 vs Collinearity')
axes[0].tick_params(axis='x', rotation=10)
axes[1].bar(df_sim['label'], df_sim['SE_b2'], color=['green','orange','red'], edgecolor='black')
axes[1].set_ylabel('SE of b2_hat')
axes[1].set_title('Standard Error of b2 vs Collinearity')
axes[1].tick_params(axis='x', rotation=10)
plt.suptitle('Multicollinearity: SEs inflate as correlation increases', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch03_mc_01_se_inflation.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Key insight: with r=0.99, SE is', round(df_sim.SE_b1.iloc[2]/df_sim.SE_b1.iloc[0], 1), 'x larger than with r=0.1')

### Step 2.3 — Monte Carlo: sampling distributions widen

In [ ]:
np.random.seed(99)
n_mc = 100
n_rep = 2000
b1_low  = np.zeros(n_rep)
b1_high = np.zeros(n_rep)
for s in range(n_rep):
    u = np.random.normal(0, 1, n_mc)
    for store, rho in [(b1_low, 0.1), (b1_high, 0.95)]:
        Sig = np.array([[1,rho],[rho,1]])
        Xc  = np.random.multivariate_normal([0,0], Sig, n_mc)
        y_  = 1 + 2*Xc[:,0] + 3*Xc[:,1] + np.random.normal(0,1,n_mc)
        Xm  = np.column_stack([np.ones(n_mc), Xc])
        b_  = np.linalg.solve(Xm.T@Xm, Xm.T@y_)
        if rho == 0.1:
            b1_low[s] = b_[1]
        else:
            b1_high[s] = b_[1]
fig, ax = plt.subplots(figsize=(9,4))
ax.hist(b1_low,  bins=60, alpha=0.6, label='Low collinearity (r=0.10)', density=True)
ax.hist(b1_high, bins=60, alpha=0.6, label='High collinearity (r=0.95)', density=True)
ax.axvline(2.0, color='black', linestyle='--', linewidth=2, label='True b1=2.0')
ax.set_xlabel('b1_hat')
ax.set_ylabel('Density')
ax.set_title('Monte Carlo: Sampling Distribution of b1 (2000 replications)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch03_mc_02_sampling_dist.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Low  collinearity: std(b1_hat) = {b1_low.std():.3f}')
print(f'High collinearity: std(b1_hat) = {b1_high.std():.3f}')

---
## 3 — Real Data Example: `wage2` Dataset

We use `wage2` which contains workers' wages, education, **and IQ scores**.
Since IQ and education are positively correlated, this is a realistic case.

In [ ]:
df = woo.dataWoo('wage2')
# Keep key variables, drop missing
df = df[['lwage','educ','IQ','exper','tenure','married','black']].dropna()
print('Shape:', df.shape)
print(df.head(8))

In [ ]:
print('Descriptive statistics:')
print(df.describe().round(2))

---
## 4 — Detect Multicollinearity

We have **four detection tools**, from quick-and-easy to most rigorous:

| Tool | What it measures | Threshold |
|---|---|---|
| Pairwise correlations | Linear association between two regressors | $|r| > 0.8$: warning |
| Variance Inflation Factor (VIF) | How much SE is inflated for each regressor | $VIF > 10$: severe |
| Condition Number | Overall collinearity of the entire $X$ matrix | $\kappa > 30$: moderate; $> 100$: severe |
| Eigenvalues of $X'X$ | Near-zero eigenvalue → near-perfect collinearity | Smallest $\approx 0$: severe |

### Step 4.1 — Pairwise Correlation Matrix

In [ ]:
vars_reg = ['educ','IQ','exper','tenure']
corr_mat = df[vars_reg].corr()
print('Pairwise correlations between regressors:')
print(corr_mat.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(6,5))
cax = ax.matshow(corr_mat, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(cax)
ax.set_xticks(range(len(vars_reg)))
ax.set_yticks(range(len(vars_reg)))
ax.set_xticklabels(vars_reg, rotation=45)
ax.set_yticklabels(vars_reg)
for i in range(len(vars_reg)):
    for j in range(len(vars_reg)):
        ax.text(j, i, f'{corr_mat.iloc[i,j]:.2f}', ha='center', va='center', fontsize=11)
ax.set_title('Correlation Matrix of Regressors', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch03_mc_03_corr_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Correlation(educ, IQ) = {corr_mat.loc["educ","IQ"]:.3f}')
print('Moderate correlation between education and IQ — potential multicollinearity')

### Step 4.2 — Variance Inflation Factor (VIF)

For regressor $x_j$:

$$VIF_j = \frac{1}{1 - R^2_j}$$

where $R^2_j$ is from regressing $x_j$ on **all other regressors**.

**Interpretation:**
- $VIF_j = 1$: no collinearity (this regressor is orthogonal to all others)
- $VIF_j = 5$: SE is $\sqrt{5} \approx 2.2$ times larger than with no collinearity
- $VIF_j = 10$: SE is $\sqrt{10} \approx 3.2$ times larger → **severe**

Note: the **SE inflation factor** is $\sqrt{VIF_j}$, not $VIF_j$ itself.

### Step 4.3 — Compute VIF manually for each regressor

In [ ]:
print('Computing VIF manually for each regressor:')
vif_manual = {}
for var in vars_reg:
    others = [v for v in vars_reg if v != var]
    formula_aux = f'{var} ~ ' + ' + '.join(others)
    R2_j = smf.ols(formula_aux, data=df).fit().rsquared
    vif_j = 1 / (1 - R2_j)
    vif_manual[var] = vif_j
    print(f'  VIF({var:8s}): R2_j={R2_j:.4f} → VIF={vif_j:.3f}  (SE factor: {vif_j**0.5:.3f}x)')

### Step 4.4 — VIF using statsmodels variance_inflation_factor

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
X_vif = sm.add_constant(df[vars_reg]).values
print('VIF from statsmodels (excludes constant):')
vif_sm = {}
for i, var in enumerate(vars_reg):
    vif_val = variance_inflation_factor(X_vif, i+1)  # +1 to skip constant
    vif_sm[var] = vif_val
    print(f'  VIF({var:8s}): {vif_val:.3f}')

### Step 4.5 — Compare manual vs statsmodels VIF

In [ ]:
print('VIF comparison:')
for var in vars_reg:
    match = np.isclose(vif_manual[var], vif_sm[var])
    print(f'  {var:8s}: manual={vif_manual[var]:.4f}, sm={vif_sm[var]:.4f}, match={match}')

### Step 4.6 — Condition Number

The **condition number** $\kappa$ of the design matrix $X$ measures overall collinearity:

$$\kappa = \sqrt{\frac{\lambda_{max}}{\lambda_{min}}}$$

where $\lambda_{max}$ and $\lambda_{min}$ are the largest and smallest eigenvalues of $X'X$.

**Rules of thumb:**
- $\kappa < 10$: no problem
- $10 \leq \kappa < 30$: moderate collinearity
- $30 \leq \kappa < 100$: strong collinearity
- $\kappa \geq 100$: severe collinearity

In [ ]:
n = len(df)
X_mat = np.column_stack([np.ones(n)] + [df[v].values for v in vars_reg])
# Condition number of X (not X'X)
cond_num = np.linalg.cond(X_mat)
print(f'Condition number of X: {cond_num:.2f}')
if cond_num < 10:
    print('  Assessment: no problematic collinearity')
elif cond_num < 30:
    print('  Assessment: moderate collinearity')
elif cond_num < 100:
    print('  Assessment: strong collinearity')
else:
    print('  Assessment: severe collinearity')

### Step 4.7 — Eigenvalues of X'X

In [ ]:
eigenvalues = np.linalg.eigvalsh(X_mat.T @ X_mat)
eigenvalues_sorted = np.sort(eigenvalues)[::-1]
print('Eigenvalues of X\'X (descending):')
for i, ev in enumerate(eigenvalues_sorted):
    print(f'  lambda_{i+1}: {ev:.4f}')
print(f'\nCondition number (sqrt(max/min)): {np.sqrt(eigenvalues_sorted[0]/eigenvalues_sorted[-1]):.2f}')
print('A near-zero eigenvalue signals near-perfect multicollinearity')

### Step 4.8 — Summary Diagnostic Table

In [ ]:
diag_df = pd.DataFrame({
    'Variable': vars_reg,
    'VIF': [vif_manual[v] for v in vars_reg],
    'SE_inflation': [vif_manual[v]**0.5 for v in vars_reg],
    'Severity': ['Low' if vif_manual[v]<5 else ('Moderate' if vif_manual[v]<10 else 'High')
                 for v in vars_reg]
})
print(diag_df.round(3).to_string(index=False))

---
## 5 — Consequences: What Goes Wrong in the Regression?

Let's run the full model and observe the symptoms of multicollinearity.

### Step 5.1 — Run OLS: lwage ~ educ + IQ + exper + tenure

In [ ]:
res_full = smf.ols('lwage ~ educ + IQ + exper + tenure', data=df).fit()
print(res_full.summary())

### Step 5.2 — Individual t-stats vs overall F-stat

In [ ]:
print('Symptom check:')
print(f'  Overall F-stat: {res_full.fvalue:.2f}  (p={res_full.f_pvalue:.4f}) → joint significance')
print(f'  R2: {res_full.rsquared:.4f}')
print('\nIndividual p-values:')
for var in ['educ','IQ','exper','tenure']:
    t  = res_full.tvalues[var]
    p  = res_full.pvalues[var]
    sig = '***' if p<0.01 else ('**' if p<0.05 else ('*' if p<0.1 else '← not significant!'))
    print(f'  {var:8s}: t={t:.3f}, p={p:.4f} {sig}')
print('\nHigh F but some insignificant t-stats → classic multicollinearity symptom')

### Step 5.3 — Wide confidence intervals

In [ ]:
ci = res_full.conf_int()
print('95% Confidence Intervals:')
for var in ['educ','IQ']:
    lo, hi = ci.loc[var]
    width = hi - lo
    print(f'  {var:6s}: [{lo:.4f}, {hi:.4f}]  width={width:.4f}')

---
## 6 — Solutions to Multicollinearity

| Solution | When to use | Trade-off |
|---|---|---|
| Drop one variable | If theoretically justified | Risk of omitted variable bias |
| Combine variables | If conceptually meaningful | Loses individual coefficients |
| Collect more data | Always helpful | Not always possible |
| Ridge regression | When prediction is the goal | Biased estimator (but lower variance) |
| Principal Components | Many correlated regressors | Harder to interpret |

### Solution A — Drop the More Collinear Variable

If the research question is primarily about the **return to education**, and IQ is just
a control, we can drop IQ. This may cause **omitted variable bias** if IQ truly belongs,
but it eliminates the collinearity problem.

**Check:** Does the educ coefficient change much? Does it become more precisely estimated?

### Step 6.1 — Model without IQ

In [ ]:
res_no_IQ = smf.ols('lwage ~ educ + exper + tenure', data=df).fit()
print('Without IQ:')
print(f'  educ:   b={res_no_IQ.params["educ"]:.4f}, SE={res_no_IQ.bse["educ"]:.4f}, t={res_no_IQ.tvalues["educ"]:.3f}')
print('\nWith IQ:')
print(f'  educ:   b={res_full.params["educ"]:.4f}, SE={res_full.bse["educ"]:.4f}, t={res_full.tvalues["educ"]:.3f}')
se_ratio = res_full.bse['educ'] / res_no_IQ.bse['educ']
print(f'\nSE ratio (with IQ / without IQ): {se_ratio:.3f}')
print('Including IQ increased the SE of educ by a factor of', round(se_ratio,2))

### Solution B — Combine Variables (Principal Component)

If `educ` and `IQ` measure a common underlying concept ('human capital'), we can
combine them into a **single index** using their first principal component.

$$PC_1 = w_1 \cdot educ_{standardised} + w_2 \cdot IQ_{standardised}$$

The weights $w_1, w_2$ maximise the variance explained by the component.

### Step 6.2 — Standardise educ and IQ

In [ ]:
df['educ_z'] = (df['educ'] - df['educ'].mean()) / df['educ'].std(ddof=1)
df['IQ_z']   = (df['IQ']   - df['IQ'].mean())   / df['IQ'].std(ddof=1)
print('Standardised variables (first 5 rows):')
print(df[['educ','educ_z','IQ','IQ_z']].head())

### Step 6.3 — PCA manually (2×2 case)

In [ ]:
Z = np.column_stack([df['educ_z'].values, df['IQ_z'].values])
C_mat = Z.T @ Z / (len(df) - 1)  # correlation matrix
print('Correlation matrix of [educ_z, IQ_z]:')
print(C_mat.round(4))

In [ ]:
eigvals, eigvecs = np.linalg.eigh(C_mat)
# Sort descending
idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]
print('Eigenvalues:', eigvals.round(4))
print('First PC loadings (educ_z, IQ_z):', eigvecs[:,0].round(4))
pct_var = eigvals[0] / eigvals.sum()
print(f'Variance explained by PC1: {100*pct_var:.1f}%')

In [ ]:
df['PC1'] = Z @ eigvecs[:,0]
print('PC1 variable added (first 5 rows):')
print(df[['educ_z','IQ_z','PC1']].head())

### Step 6.4 — Regress on PC1 instead of educ + IQ

In [ ]:
res_pca = smf.ols('lwage ~ PC1 + exper + tenure', data=df).fit()
print(res_pca.summary())
print(f'PC1 VIF:')
X_pca = sm.add_constant(df[['PC1','exper','tenure']]).values
from statsmodels.stats.outliers_influence import variance_inflation_factor
for i, lbl in enumerate(['PC1','exper','tenure']):
    print(f'  VIF({lbl}): {variance_inflation_factor(X_pca, i+1):.3f}')

### Solution C — Ridge Regression

**Ridge regression** adds a penalty $\lambda \|\beta\|^2$ to the OLS objective:

$$\hat{\beta}_{Ridge} = \arg\min_\beta \left[ (y - X\beta)'(y-X\beta) + \lambda \beta'\beta \right]$$

**Closed-form solution:**

$$\hat{\beta}_{Ridge} = (X'X + \lambda I)^{-1} X'y$$

**Intuition:** Adding $\lambda I$ to $X'X$ 'stabilises' it even when $X'X$ is near-singular.

**Trade-off:** Ridge is **biased** but has **lower variance** than OLS when collinearity is severe.
It is NOT the right choice if unbiasedness is required. It is useful for **prediction**.

### Step 6.5 — Ridge regression manually

In [ ]:
# Standardise all regressors first (ridge needs standardised inputs)
reg_vars = ['educ','IQ','exper','tenure']
X_raw = np.column_stack([df[v].values for v in reg_vars])
X_std = (X_raw - X_raw.mean(axis=0)) / X_raw.std(axis=0, ddof=1)
y_arr = df['lwage'].values
n_r   = len(y_arr)
print('Standardised X (first 3 rows):')
print(X_std[:3].round(4))

In [ ]:
lam = 10.0  # ridge penalty
XtX_ridge = X_std.T @ X_std + lam * np.eye(X_std.shape[1])
Xty_ridge  = X_std.T @ y_arr
beta_ridge = np.linalg.solve(XtX_ridge, Xty_ridge)
print(f'Ridge coefficients (lambda={lam}):')
for var, b in zip(reg_vars, beta_ridge):
    print(f'  {var:8s}: {b:.5f}')

### Step 6.6 — Compare OLS vs Ridge coefficients (standardised)

In [ ]:
XtX_ols = X_std.T @ X_std
beta_ols_std = np.linalg.solve(XtX_ols, X_std.T @ y_arr)
print('Comparison: standardised OLS vs Ridge')
print(f'{"Variable":10s}  {"OLS":>10s}  {"Ridge":>10s}  {"Shrinkage":>10s}')
for var, b_ols, b_rid in zip(reg_vars, beta_ols_std, beta_ridge):
    shrink = 100*(b_ols - b_rid)/abs(b_ols) if b_ols != 0 else 0
    print(f'{var:10s}  {b_ols:10.5f}  {b_rid:10.5f}  {shrink:9.1f}%')

### Step 6.7 — Ridge path: how coefficients shrink as λ increases

In [ ]:
lambdas = np.logspace(-2, 3, 100)
ridge_coefs = np.zeros((len(lambdas), len(reg_vars)))
for i, lam_i in enumerate(lambdas):
    B = np.linalg.solve(X_std.T@X_std + lam_i*np.eye(4), X_std.T@y_arr)
    ridge_coefs[i] = B
fig, ax = plt.subplots(figsize=(9,5))
for j, var in enumerate(reg_vars):
    ax.semilogx(lambdas, ridge_coefs[:,j], linewidth=2, label=var)
ax.axvline(10, color='gray', linestyle='--', alpha=0.6, label='lambda=10')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Ridge penalty lambda (log scale)')
ax.set_ylabel('Standardised coefficient')
ax.set_title('Ridge Path: Coefficients shrink to 0 as lambda increases')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch03_mc_04_ridge_path.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 7 — Verify the Fix: Re-run Diagnostics After Each Solution

After applying a solution, **always re-check VIF** to confirm collinearity is reduced.

### Step 7.1 — VIF after dropping IQ

In [ ]:
vars_noIQ = ['educ','exper','tenure']
print('VIF after dropping IQ:')
for var in vars_noIQ:
    others = [v for v in vars_noIQ if v != var]
    R2_j = smf.ols(f'{var} ~ ' + ' + '.join(others), data=df).fit().rsquared
    vif_j = 1 / (1 - R2_j)
    print(f'  VIF({var:8s}): {vif_j:.3f}')

### Step 7.2 — VIF after using PC1

In [ ]:
vars_pca = ['PC1','exper','tenure']
print('VIF after replacing educ+IQ with PC1:')
for var in vars_pca:
    others = [v for v in vars_pca if v != var]
    R2_j = smf.ols(f'{var} ~ ' + ' + '.join(others), data=df).fit().rsquared
    vif_j = 1 / (1 - R2_j)
    print(f'  VIF({var:8s}): {vif_j:.3f}')

### Step 7.3 — Comparing solutions: SE of key coefficients

In [ ]:
print('Effect of each solution on SE of education-related coefficient:')
print(f'  Full model (educ+IQ):     SE(educ) = {res_full.bse["educ"]:.4f}')
print(f'  Drop IQ:                  SE(educ) = {res_no_IQ.bse["educ"]:.4f}')
print(f'  PCA (PC1):                SE(PC1)  = {res_pca.bse["PC1"]:.4f}')
print(f'\nAll three solutions reduce SE compared to the full collinear model')

---
## 8 — Decision Guide: Which Solution to Use?

```
         Is one variable theoretically redundant?
                    ↓              ↓
                  YES              NO
                   ↓               ↓
           Drop the variable   Are variables measuring same concept?
           (beware OVB)               ↓           ↓
                                    YES           NO
                                     ↓             ↓
                                Combine into    Primary goal?
                                index / PCA      ↓        ↓
                                            Inference  Prediction
                                                ↓           ↓
                                         Get more data  Ridge/LASSO
```

**Key principle:** Multicollinearity is a **data problem**, not a model error.
The solution must be guided by **theory and research question**, not just statistics.

In [ ]:
final_table = pd.DataFrame({
    'Model': ['Full (educ+IQ+exper+tenure)', 'Drop IQ (educ+exper+tenure)', 'PCA (PC1+exper+tenure)'],
    'VIF_educ_or_PC1': [
        vif_manual['educ'],
        1/(1-smf.ols('educ~exper+tenure',data=df).fit().rsquared),
        1/(1-smf.ols('PC1~exper+tenure',data=df).fit().rsquared)
    ],
    'SE_key': [res_full.bse['educ'], res_no_IQ.bse['educ'], res_pca.bse['PC1']],
    'R2':     [res_full.rsquared, res_no_IQ.rsquared, res_pca.rsquared]
})
print(final_table.round(4).to_string(index=False))
final_table.to_csv(os.path.join(output_dir,'table_ch03_mc_solutions.csv'), index=False)
print('\nSaved table_ch03_mc_solutions.csv')